In [0]:
from pyspark.sql import functions as F

In [0]:
# Objetivo: alimentar modelos de pricing e previsão de demanda

silver_events   = spark.table("silver.events")
fact_rev        = spark.table("gold_bi.fact_event_revenue")
fact_tickets    = spark.table("gold_bi.fact_ticket_sales")

# Histórico de eventos por organização
org_history = (
    fact_rev
    .filter(F.col("status") == "completed")
    .groupBy("org_id")
    .agg(
        F.count("event_id").alias("org_events_count"),
        F.avg("gross_ticket_revenue_brl").alias("org_avg_revenue"),
        F.avg("roi_ratio").alias("org_avg_roi"),
        F.avg("occupancy_rate").alias("org_avg_occupancy"),
    )
)

features_event = (
    silver_events.alias("e")
    .join(fact_rev.alias("r"), "event_id", "left")
    .join(org_history.alias("oh"), F.col("e.org_id") == F.col("oh.org_id"), "left")
    .select(
        F.col("e.event_id"),
        F.col("e.org_id"),
        # Features temporais
        F.dayofweek("e.start_date").alias("start_day_of_week"),
        F.month("e.start_date").alias("start_month"),
        F.hour("e.start_date").alias("start_hour"),
        F.col("e.duration_seconds"),
        # Features de capacidade
        F.col("e.max_attendees"),
        F.col("venue_capacity"),
        F.col("e.is_online").cast("int"),
        # Features de ocupação/resultado (label no treino, feature na inferência pré-evento)
        F.col("r.occupancy_rate"),
        F.col("r.gross_ticket_revenue_brl"),
        F.col("r.roi_ratio"),
        # Features do histórico da organização (contexto de maturidade)
        F.col("oh.org_events_count"),
        F.col("oh.org_avg_revenue"),
        F.col("oh.org_avg_roi"),
        F.col("oh.org_avg_occupancy"),
        # Target (para treino de pricing)
        F.col("r.avg_ticket_price_cents"),
    )
)

(features_event.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_ai.features_event_profile"))